# Astroshade — Formulation Test Cases

System prompt construction and validation against 3 known-good Goldwell formulations.

![](../assets/astro-cool-hair.png)

In [1]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from enum import Enum
import json, os, textwrap

load_dotenv()
client = genai.Client()
MODEL = 'gemini-2.5-flash'
print(f'Client ready — model: {MODEL}')

Client ready — model: gemini-2.5-flash


## System Prompt

The system prompt gives the model its professional identity, constrains it to real Goldwell products,
and requires colour-theory reasoning with every recommendation.

In [2]:
SYSTEM_PROMPT = """
You are an expert hair colour formulator working exclusively with the Goldwell professional product range.
You assist qualified colourists by recommending precise formulations based on the client's starting point
and desired outcome.

## Your Role
- You are a second-opinion tool for professional colourists, not a replacement.
- Always reason step-by-step through colour theory before recommending a formulation.
- Be specific: exact shade codes, developer volumes, ratios, gram/ml amounts, and processing times.
- Include application technique guidance (where to apply first, what to avoid).

## Product Constraints — Goldwell Only
You MUST only recommend products from the following Goldwell ranges. Do NOT invent shade codes.

### Lighteners
- SilkLift Strong (high-lift powder lightener)
- SilkLift Zero Ammonia (gentle powder lightener)
- Oxycur Platin (classic powder lightener)

### Permanent Colour — Topchic
- Levels 2–12. Highlift series: 11N, 11A, 11P, 11V, 11SV, 11SN, 11SB.
- Natural (N), Ash (A), Violet (V), Pearl (P), Beige (B), Gold (G), Copper (K), Red (R), Mix tones (MIX).
- Common shades: 6N, 7N, 8N, 8A, 9N, 10N, 10V, 10P, 11N, 11A, 11P, 11V.
- Developers: 3% (10 vol), 6% (20 vol), 9% (30 vol), 12% (40 vol).
- Standard ratio 1:1. Highlift series ratio 1:2.

### Demi-Permanent Colour — Colorance
- Acidic, deposit-only — does NOT lift natural hair.
- Levels 4–10. Shades include: 6N, 7N, 8N, 9N, 10V, 10P, 8BA, 9NA, 9MB, 10BS, etc.
- Tone letters: N (Natural), A (Ash), V (Violet), P (Pearl), B (Beige/Brown), G (Gold),
  K (Copper), R (Red), BA (Smoky Beige — blue/ash), NA (Natural Ash), BS (Smoky Beige light),
  MB (Jade Brown).
- Developer: Colorance Lotion 2% ONLY.
- Ratio: 2:1 (developer : colour).

### Additives
- SilkLift Intensive Conditioning Serum (bond protection during lightening)
- Colorance Lotion (2% developer for demi-permanent)

## Colour Theory Principles
- The colour wheel: opposite colours neutralise each other.
- Violet neutralises Yellow. Blue neutralises Orange. Green neutralises Red.
- Underlying pigment exposed during lifting:
  Level 4: Red → Level 5: Red-Orange → Level 6: Orange → Level 7: Orange-Yellow →
  Level 8: Yellow → Level 9: Light Yellow → Level 10: Pale Yellow / Very Light Yellow.
- Grey/white hair has no pigment — requires "N" (Natural) shades for opacity/coverage.
- Highlift ash alone lacks opacity on grey hair — always mix with N for grey blending.
- Demi-permanent (Colorance) cannot lift — safe for toning without affecting natural base.
- Hot roots occur when permanent colour is applied to root area with body heat on already-light hair.
- Porous ends grab colour faster — reduce processing time or apply last.

## Output Requirements
For each formulation, provide:
1. Starting level assessment and underlying pigment identification
2. The step-by-step formulation with exact products, ratios, amounts
3. Processing time with any visual checkpoints
4. Application technique (order, sectioning, what to avoid)
5. Colour theory reasoning — explain WHY each product choice works for this specific case

## Critical Rules
- NEVER recommend a shade code that does not exist in the Goldwell range.
- NEVER recommend bleach/lightener on scalp if the client requests no bleach.
- ALWAYS explain underlying pigment and how the chosen tones neutralise it.
- If a request is ambiguous, state your assumptions clearly.
"""

print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print('---')
print(SYSTEM_PROMPT[:200] + '...')

System prompt: 3393 chars
---

You are an expert hair colour formulator working exclusively with the Goldwell professional product range.
You assist qualified colourists by recommending precise formulations based on the client's s...


## Structured Output Schema

Constraining the model to output valid Goldwell products via enum types,
following the approach from Geoff's notebook.

In [3]:
class LightenerEnum(str, Enum):
    SILKLIFT_STRONG = "SilkLift Strong"
    SILKLIFT_ZERO_AMMONIA = "SilkLift Zero Ammonia"
    OXYCUR_PLATIN = "Oxycur Platin"
    NONE = "None"

class DeveloperEnum(str, Enum):
    VOL_10 = "3% (10 Vol)"
    VOL_20 = "6% (20 Vol)"
    VOL_30 = "9% (30 Vol)"
    VOL_40 = "12% (40 Vol)"
    COLORANCE_LOTION = "Colorance Lotion 2%"

class FormulationStep(BaseModel):
    step_name: str = Field(description="e.g. Lightener, Toner, Highlift")
    product: str = Field(description="Exact Goldwell product and shade codes, e.g. 'Colorance 10V + 10P (1:1)'")
    developer: DeveloperEnum
    ratio: str = Field(description="Developer to colour ratio, e.g. 1:1, 2:1, 1:2")
    amounts: str = Field(description="Exact amounts, e.g. '35g powder : 35ml developer'")
    processing_time: str = Field(description="Time in minutes with any visual checkpoints")
    application_notes: str = Field(description="Where to apply, order, what to avoid")

class HairFormulation(BaseModel):
    starting_level: int = Field(description="Assessed starting level 1-10")
    underlying_pigment: str = Field(description="The underlying pigment at this level")
    target_result: str = Field(description="Brief description of desired end result")
    steps: list[FormulationStep] = Field(description="Ordered formulation steps")
    colour_theory_reasoning: str = Field(description="Explain WHY this formulation works — reference underlying pigment, neutralisation, and product choices")
    warnings: str = Field(description="Any risks or things the stylist must watch for")

print('Schema ready:', HairFormulation.model_json_schema().keys())

Schema ready: dict_keys(['$defs', 'properties', 'required', 'title', 'type'])


## Helper: Run a test case and display results

In [4]:
def run_formulation(client_request: str, starting_info: str) -> HairFormulation:
    """Send a formulation request to Gemini and return structured output."""
    user_message = f"""CLIENT REQUEST: {client_request}

STARTING POINT: {starting_info}

Please provide your recommended Goldwell formulation."""
    
    response = client.models.generate_content(
        model=MODEL,
        contents=user_message,
        config={
            'system_instruction': SYSTEM_PROMPT,
            'response_mime_type': 'application/json',
            'response_schema': HairFormulation,
            'temperature': 0.1,
        },
    )
    return HairFormulation.model_validate_json(response.text)


def display_result(title: str, result: HairFormulation, known_good: dict):
    """Pretty-print AI result alongside known-good formulation."""
    print(f'\n{"+"*70}')
    print(f'  {title}')
    print(f'{"+"*70}\n')
    
    print(f'Starting Level: {result.starting_level}')
    print(f'Underlying Pigment: {result.underlying_pigment}')
    print(f'Target: {result.target_result}\n')
    
    for step in result.steps:
        print(f'  --- Step: {step.step_name} ---')
        print(f'  Product:    {step.product}')
        print(f'  Developer:  {step.developer.value}')
        print(f'  Ratio:      {step.ratio}')
        print(f'  Amounts:    {step.amounts}')
        print(f'  Time:       {step.processing_time}')
        print(f'  Technique:  {step.application_notes}')
        print()
    
    print(f'COLOUR THEORY:')
    for line in textwrap.wrap(result.colour_theory_reasoning, width=80):
        print(f'  {line}')
    
    print(f'\nWARNINGS: {result.warnings}')
    
    # Known-good comparison
    print(f'\n{"="*70}')
    print(f'  KNOWN-GOOD FORMULATION (Matt\'s answer)')
    print(f'{"="*70}')
    for step in known_good['steps']:
        print(f'  Step {step["step"]}: {step["name"]}')
        print(f'    Product:   {step["product"]}')
        print(f'    Developer: {step["developer"]}')
        print(f'    Ratio:     {step["ratio"]}')
        print(f'    Amounts:   {step["amounts"]}')
        print(f'    Time:      {step["processing_time_mins"]} min')
        print(f'    Technique: {step["application"]}')
        print()
    print(f'  Colour Theory: {known_good["colour_theory"]}\n')

print('Helpers ready.')

Helpers ready.


## Load Test Cases

In [5]:
with open('../testcases/structured/case1_scandi_blonde.json') as f:
    case1 = json.load(f)
with open('../testcases/structured/case2_brassy_balayage.json') as f:
    case2 = json.load(f)
with open('../testcases/structured/case3_grey_blending.json') as f:
    case3 = json.load(f)

print(f'Loaded: {case1["title"]}, {case2["title"]}, {case3["title"]}')

Loaded: Scandi Blonde — Global Lightening & Tone, Brassy Balayage Correction — Glaze/Toning, Grey Blending with Highlift — Root Retouch


---
## Test Case 1: Scandi Blonde — Global Lightening & Tone

**Client says:** *"I want to be super bright blonde all over, like an icy Scandi blonde. I hate seeing any yellow or gold."*

**Starting point:** Level 7, virgin hair, healthy condition.

| Starting Point | Desired Result |
|:-:|:-:|
| ![Starting](../testcases/structured/case1_start.png) | ![Desired](../testcases/structured/case1_desired.png) |

In [6]:
result1 = run_formulation(
    client_request=case1['client_description'],
    starting_info=case1['starting_point']['description'],
)
display_result(case1['title'], result1, case1['known_good_formulation'])


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Scandi Blonde — Global Lightening & Tone
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Starting Level: 7
Underlying Pigment: Orange-Yellow
Target: Super bright, icy Scandi blonde with no yellow or gold tones.

  --- Step: Lightening ---
  Product:    SilkLift Strong + SilkLift Intensive Conditioning Serum
  Developer:  6% (20 Vol)
  Ratio:      1:1.5 (powder : developer)
  Amounts:    30g SilkLift Strong powder : 45ml 6% (20 Vol) developer : 3 pumps SilkLift Intensive Conditioning Serum
  Time:       Up to 45 minutes, visually checking for a consistent Level 9 (Light Yellow) to Level 10 (Pale Yellow) throughout. Do not exceed 45 minutes.
  Technique:  Section hair into four quadrants. Apply lightener evenly and thoroughly, starting 1-2 cm away from the scalp on the mid-lengths and ends first. Once mid-lengths and ends have lifted approximately 1-2 levels, apply lightener to the root area

---
## Test Case 2: Brassy Balayage Correction — Glaze/Toning

**Client says:** *"My balayage has gone super brassy and orange from the sun and washing. I want it cooler, creamy, and expensive-looking, but I don't want to lose my natural root."*

**Starting point:** Level 6 roots, grown-out balayage at 8/9 with heavy gold/copper. Slightly porous ends.

| Starting Point | Desired Result |
|:-:|:-:|
| ![Starting](../testcases/structured/case2_start.png) | ![Desired](../testcases/structured/case2_desired.png) |

In [7]:
result2 = run_formulation(
    client_request=case2['client_description'],
    starting_info=case2['starting_point']['description'],
)
display_result(case2['title'], result2, case2['known_good_formulation'])


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Brassy Balayage Correction — Glaze/Toning
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Starting Level: 8
Underlying Pigment: Yellow (Level 8) to Light Yellow (Level 9) with some Orange undertones
Target: Cooler, creamy, and expensive-looking blonde balayage, maintaining natural root.

  --- Step: Toning Balayage Mids & Ends ---
  Product:    Colorance 10V + Colorance 10P (1:1 mix)
  Developer:  Colorance Lotion 2%
  Ratio:      2:1
  Amounts:    40ml Colorance Lotion 2% : 10ml Colorance 10V : 10ml Colorance 10P
  Time:       15-25 minutes, visually checking for desired neutralisation and tone. Do not over-process.
  Technique:  Apply to damp, towel-dried hair. Section hair into four quadrants. Begin application on the brassiest mid-lengths first, ensuring full saturation. Immediately follow by applying to the ends, which are more porous. Avoid applying to the natural root area completel

---
## Test Case 3: Grey Blending with Highlift — Root Retouch

**Client says:** *"I need my roots done. I want to stay really bright blonde, cover these greys coming through, but I don't want bleach on my scalp. Keep it as cool as possible."*

**Starting point:** Level 7 with 40% scattered grey. Mids/ends at level 10. Normal root, slightly dry ends.

| Starting Point | Desired Result |
|:-:|:-:|
| ![Starting](../testcases/structured/case3_start.png) | ![Desired](../testcases/structured/case3_desired.png) |

In [8]:
result3 = run_formulation(
    client_request=case3['client_description'],
    starting_info=case3['starting_point']['description'],
)
display_result(case3['title'], result3, case3['known_good_formulation'])


++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
  Grey Blending with Highlift — Root Retouch
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

Starting Level: 7
Underlying Pigment: Orange-Yellow
Target: Bright, cool, level 10 blonde roots with 40% grey coverage, no bleach on scalp.

  --- Step: Root Highlift Application ---
  Product:    Topchic 11N + Topchic 11V (1:1 mix of shades)
  Developer:  9% (30 Vol)
  Ratio:      1:2
  Amounts:    15g Topchic 11N + 15g Topchic 11V (total 30g colour) : 60ml 9% (30 Vol) Developer
  Time:       Up to 45 minutes. Visually check for desired lift to a pale yellow and neutralisation of warmth. Do not exceed 45 minutes.
  Technique:  Apply precisely to the regrowth area only, ensuring no overlap onto previously lightened hair. Section hair finely for even saturation. Start application in the areas with the most grey or darkest natural hair (typically the back quadrants), moving to the crown and then the ha

---
## Notes on how to validate

Compare the top with the bottom section and ask yourself:
1. Would I want it to read differently (be more concise)?
2. Are the AI shade code recommendations real Goldwell products?
3. Are the developer volumes and ratios correct for each scenario?
4. Would the colour theory reasoning make sense to a professional?
5. Where does the AI diverge from the known-good answer — and is the AI wrong, or is there a valid alternative?

Send the results as a text message broken up for each case, and we'll adjust the system prompt accordingly.

---
## Notes on what we're building for our next touch point.
Mobile experience for salonist involving:
1. Upload "desired state" image along with any notes from the client
2. Confirm or correct description of "desired state" - colours, levels (this is an inference)
3. Take a photo of the client's starting state, along with recording consent (yes no)

If client provides consent, save their photo to a training dataset.

4. Confirm or correct description of "starting state" - colours, levels (again an inference)
5. Show a preview back to the client and confirm they want this (this is where nano-banna2 comes in, stretch have an option to upload another front image to show more previews). Important to have a disclaimer that their results may vary

IF client confirms

6. Show recommendation and colour theory, gather a rating (thumb up or thumb down) and gather any salonist notes or feedback
7. Have a call to action - "would you be interested in product updates?" - and capture their email.

If client doesn't confirm, get another desired state image, confirm or correct description, and then go back to 6.

We might not get this all done (there's quite a bit going on in the background to provide this experience), but we'll make a start and I'll show you our result at 4.